# Model AI phân loại rác sử dụng hình ảnh

# Import Thư Viện

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

# Chuẩn bị dữ liệu

In [ ]:
from pathlib import Path
DATA_DIR = Path(r"E:\SPCK-CSI18\data\dataset")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
VAL_DIR = DATA_DIR / "val"

# Chuẩn hóa dữ liệu

In [3]:
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 0.001
IMG_SIZE = (256, 256)
NUM_CLASSES = 10
WEIGHT_DECAY = 0.05

In [ ]:
from torchvision import transforms
from torchvision import datasets


train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor()
])


train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)


print(train_dataset.classes)
print(len(train_dataset.classes))

['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']
10


# ABCXYZ

In [5]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [6]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([16, 3, 256, 256])
torch.Size([16])


# Model

In [8]:
model = models.mobilenet_v3_small(weights="DEFAULT")
print(type(model))

<class 'torchvision.models.mobilenetv3.MobileNetV3'>


In [13]:
model.classifier[-1] = nn.Linear(
    model.classifier[-1].in_features,
    NUM_CLASSES
)
print(model.classifier)
print(model.classifier[-1])

Sequential(
  (0): Linear(in_features=576, out_features=1024, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1024, out_features=10, bias=True)
)
Linear(in_features=1024, out_features=10, bias=True)


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
for param in model.parameters():
    print(param.shape)
    break

torch.Size([16, 3, 3, 3])


# Training

In [16]:
EPOCHS = 1

for epoch in range(EPOCHS):

    model.train()

    for images, labels in train_loader:

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {loss.item():.4f}")

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "garbage_classifier.pth")
print("Model saved!")